In [ ]:
#| default_exp search

# Search

> MCTS API. Monte Carlo Tree Search is a random sampling algorithm. AlphaZero modifies MCTS to employ NNATS/DLATS: deeplearning augmented tree search. 

In [ ]:
#|hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np
import alphazero.go as go
import torch.nn

MCTS uses an `MCSTree` which manages `MCSTNode`s.

In [ ]:
#| export
class MCSTNode():
    """
    Represents a single node in a search tree.
    """
    def __init__(self, 
                 state, # state engine object. Handles scoring, valid actions, turn, etc.
                 action_probs:np.ndarray=None, # probabilities on taking each action. Initially output from a model, can be modified by AlphaZero's MCTS.
                 action_index:int=None, # index into the previous node state's action array leading to the current node. Used to lookup the probability of the action that led to this node.
                 branches:tuple=None, # n_actions-length array containing indices of branch nodes. Branches are reserved indices and do not have to exist in the tree. Node branch indices must be reserved for a 1-1 action-branch node mapping.
                 descendants:list=None, # index array of all nodes descending from current. Used to quickly prune the tree. Descendants must exist in the tree.
                 value:float=None, # action value of the current state
                 parent_index:int=None, # index of parent node
                 index:int=None,): # node index in search tree
        self.state = state
        self.action_probs = action_probs
        self.action_index = action_index
        self.branches = branches
        self.descendants = descendants
        self.visits = 0
        self.value = value
        self.path_value = value
        self.parent_index = parent_index
        self.index = index

    # def update(self):
    #     """
    #     Updates a node's statistics.
    #     """
    #     self.mean_action_value = self.path_value / self.visits
    #     self.exploration_value = self.

`MCSTNode.path_value` stores the total action value of the node for all paths taken through it. This is the sum of the current node, and all nodes it leads to.

`MCSTNode.visits` stores the number of times MCTS has visited a node.

In [ ]:
#| export
class MCSTree():
    """
    Monte Carlo Search Tree class. An MCS tree is functionally a dictionary of index:`MCTSNode` pairs, with attendant methods for simulation and maintenance. The MCST binds a model with a state engine.
    """
    def __init__(self, 
                 n_actions:int=None, # number of actions at each turn
                 n_sims:int=1, # number of simulations to run per turn. Tree will expand by this many nodes each turn on average.
                 model:torch.nn.Module=None, # the neural network model which assesses states.
                 noise_ε:float=0.25, # epsilon parameter modifying dirichlet noise function
                 noise_α:float=0.03, # alpha parameter for dirichlet noise function
                 c_puct:float=1.0, # MCTS exploration parameter constant.
                 ):
        self.rng = np.random.default_rng(seed=0)
        self.tree = {}
        self.n_actions = n_actions
        self.model = model
        self.n_sims = n_sims
        self.noise_ε =  noise_ε
        self.noise_α = noise_α
        self.root_idx = 0
        self.next_idx = 1
        self.add_node(None, self.root_idx, go.Position())
        self.c_puct = c_puct
        self.temperature_inv = 1/1.0
        self.moves_played = 0
        self.policy = np.zeros(n_actions)

    def _populate_descendants(self, idx):
        """
        Adds the current node to list of descendants of all ascendant nodes to root.
        """
        parent_idx = self.tree[idx].parent_index
        while parent_idx in self.tree:
            self.tree[parent_idx].descendants.append(idx)
            parent_idx = self.tree[parent_idx].parent_index

    def _build_node(self, state):
        raise NotImplementedError("Method is currently a stub.")

    def _remove_node(self, idx): # NOTE: redundant?
        del self.tree[idx]

    def assess_state(self, state):
        """
        Runs a model to assess a state, and post processes the result. Returns an action value and array of probabilities.
        """
        # run model on state
        if self.model is None:
            value = self.rng.random()
            probs = self.rng.random(size=state.all_legal_moves().size) * state.all_legal_moves()
        else:
            value, probs = self.model(state.board)
        # apply noise: P(s,) = (1 - ε)•p(s,) + ε•η; η=dir(α)
        probs = (1 - self.noise_ε)*probs +  self.noise_ε*self.rng.dirichlet(self.noise_α*np.ones_like(probs))
        return value * state.to_play * -1, probs * state.to_play # state value from perspective of player that led to this state; action value from current player

    def add_node(self,
                 parent_idx, # tree index of parent node
                 idx,        # tree index of node
                 state):     # state engine instance
        state_value,state_probs = self.assess_state(state) # probabilities & values are computed once on node creation
        self.tree[idx] = MCSTNode(state=state,
                    action_probs=state_probs,
                    branches = tuple(i + self.next_idx for i in range(state.all_legal_moves().sum())),
                    descendants = [],
                    value=state_value,
                    parent_index=parent_idx,
                    index=idx)
        self.next_idx += len(self.tree[idx].branches)
        self._populate_descendants(idx)

    def prune(self, idx):
        """
        Removes a node and all its descendants from the search tree. Resets the root node if specified.
        """
        for i in self.tree[idx].descendants: del self.tree[idx]
        del self.tree[idx]

    def set_root_and_prune(self, idx): # NOTE: may need a less ambiguous name for this. too similar to simulation selection (which isn't destructive)
        """
        Sets tree root to selected node, removes previous root, and prunes all nodes not descended from the new root.
        """
        for branch in self.tree[self.root_idx].branches: # remove all subtrees except selected
            if branch != idx: self.prune(idx)
        self._remove_node(self.root_idx)                 # remove root node
        self.root_idx = idx                              # set root node to selected

    def search(self):
        """
        MCTS simulation loop.
        """
        # run simulations
        for i in range(self.n_sims): self.simulate()
        # build policy array
        for i,bdx in enumerate(self.tree[self.root_idx].branches):
            if bdx in self.tree: self.policy[i] += self.tree[bdx].visits
        exposum = self.policy.sum() ** (self.temperature_inv)
        self.policy = np.array([pi**(self.temperature_inv)/exposum for pi in self.policy])
        
        self.moves_played += 1
        if self.moves_played == 30: self.temperature_inv = 1/0.1

        # return selected action
        return np.argmax(self.policy)

    def simulate(self):
        """
        Implements a = argmax(Q(s,a), U(s,a))
        """

        # forloop
        idx = self.root_idx
        parent_idx = self.tree[idx].parent_index
        adx = self.n_actions # default action is pass

        # explore tree until reaching end state or unexplored node
        while idx in self.tree and not self.tree[idx].state.is_game_over():
            self.tree[idx].visits += 1

            # upper confidence bound = mean action value + exploration value; UCB = Q+U
            ucbs = np.zeros(self.n_actions, dtype=np.float32)
            sum_visits = sum(self.tree[bdx].visits for bdx in self.tree[idx].branches if bdx in self.tree)
            for i in range(self.tree[idx].state.all_legal_moves().sum()): # node branch idxs must be reserved for a 1-1 action-branch node mapping
                # build UCB: Q(s,a) + U(s,a)

                bdx = self.tree[idx].branches[i] # branch index
                if bdx in self.tree:
                    mav = self.tree[bdx].path_value / self.tree[bdx].visits
                    exv = self.c_puct * self.tree[idx].action_probs[i] * np.sqrt(sum_visits - self.tree[bdx].visits) / (1 + self.tree[bdx].visits)
                else:
                    mav = exv = 0.0
                ucbs[i] = (mav + exv) * self.tree[idx].state.to_play # action values seen from current player's perspective
            
            # select next action: a = argmax(UCB)
            # omit all invalid moves from consideration via mask
            mask = np.zeros(self.n_actions)
            mask[np.where(self.tree[idx].state.all_legal_moves()==0)] = 1
            adx = np.argmax(np.ma.array(ucbs, mask=mask)).astype(int)
            parent_idx = idx
            idx = self.tree[idx].branches[adx]
            
        # if new state, create node and assess
        if idx not in self.tree:
            coord = go.action_array_index_to_coord(adx) # convert action array index into playable coordinate
            new_state = self.tree[parent_idx].state.play_move(coord)
            self.add_node(parent_idx=parent_idx, idx=idx, state=new_state)
            self.tree[idx].visits += 1

        # propagate action value back down path
        value = self.tree[idx].value
        idx = self.tree[idx].parent_index
        while idx != self.root_idx:
            value *= -1 # sign flip for alternative player turn
            self.tree[idx].path_value += value
            idx = self.tree[idx].parent_index

        # TODO: heavier weighting for win/loss?


## MCTS Example

In [ ]:
mctree = MCSTree(n_actions=362, n_sims=100)

In [ ]:
%%time
move_idx = mctree.search()

CPU times: user 669 ms, sys: 4.54 ms, total: 673 ms
Wall time: 673 ms


In [ ]:
mctree.policy

array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [ ]:
mctree.tree[0].action_probs[:20]

array([0.20234004, 0.03073014, 0.01239573, 0.60995311, 0.68456668,
       0.45497683, 0.54712242, 0.40772146, 0.70130432, 0.61189017,
       0.00205388, 0.64309569, 0.02518946, 0.5473042 , 0.13174172,
       0.64738419, 0.40609772, 0.22478392, 0.31701542, 0.02123975])

In [ ]:
mctree.tree[0].value

-0.6369616873214543

In [ ]:
mctree.next_idx

32128

In [ ]:
mctree.tree[0].descendants[:12]

[1, 364, 726, 1087, 1447, 1806, 2164, 2521, 2877, 3232, 3586, 3939]

In [ ]:
[mctree.tree[i].visits for i in mctree.tree[0].descendants[:12]]

[100, 99, 98, 97, 96, 95, 94, 93, 92, 91, 90, 89]

In [ ]:
engine = go.Position()
engine = engine.play_move(go.action_array_index_to_coord(move_idx))

In [ ]:
print(engine)

   A B C D E F G H J K L M N O P Q R S T   
19 X<. . . . . . . . . . . . . . . . . . 19
18 . . . . . . . . . . . . . . . . . . . 18
17 . . . . . . . . . . . . . . . . . . . 17
16 . . . . . . . . . . . . . . . . . . . 16
15 . . . . . . . . . . . . . . . . . . . 15
14 . . . . . . . . . . . . . . . . . . . 14
13 . . . . . . . . . . . . . . . . . . . 13
12 . . . . . . . . . . . . . . . . . . . 12
11 . . . . . . . . . . . . . . . . . . . 11
10 . . . . . . . . . . . . . . . . . . . 10
 9 . . . . . . . . . . . . . . . . . . .  9
 8 . . . . . . . . . . . . . . . . . . .  8
 7 . . . . . . . . . . . . . . . . . . .  7
 6 . . . . . . . . . . . . . . . . . . .  6
 5 . . . . . . . . . . . . . . . . . . .  5
 4 . . . . . . . . . . . . . . . . . . .  4
 3 . . . . . . . . . . . . . . . . . . .  3
 2 . . . . . . . . . . . . . . . . . . .  2
 1 . . . . . . . . . . . . . . . . . . .  1
   A B C D E F G H J K L M N O P Q R S T   
Move: 1. Captures X: 0 O: 0



## Concept Notes

**Policy**

...

[[ref](https://ai.stackexchange.com/questions/27954/how-does-policy-network-learn-in-alphazero)]

In [ ]:
sample_policy = np.random.default_rng(0).integers(0,20,10)
exposum = sample_policy.sum() ** (1/.1)
print(sample_policy)
sample_policy_t0 = [np.format_float_scientific(spi**(1/.1)/exposum,5) for spi in sample_policy]
print(exposum)
print(sample_policy_t0, np.argmax(sample_policy_t0))
exposum = sample_policy.sum() ** (1/1.)
sample_policy_t1 = [np.format_float_scientific(spi**(1/1.)/exposum,5) for spi in sample_policy]
print(exposum)
print(sample_policy_t1, np.argmax(sample_policy_t1))

[17 12 10  5  6  0  1  0  3 16]
2.82475249e+18
['7.13689e-07', '2.19196e-08', '3.54013e-09', '3.45716e-12', '2.14058e-11', '0.e+00', '3.54013e-19', '0.e+00', '2.09041e-14', '3.89242e-07'] 0
70.0
['2.42857e-01', '1.71429e-01', '1.42857e-01', '7.14286e-02', '8.57143e-02', '0.e+00', '1.42857e-02', '0.e+00', '4.28571e-02', '2.28571e-01'] 4


**Noise**

Dirichlet Noise takes an array of parameters and produces a dirichlet distribution. In AlphaZero's case the array is the same size as the action probabilities, and consists of the same parameter. [[ref](https://stats.stackexchange.com/questions/322831/purpose-of-dirichlet-noise-in-the-alphazero-paper)]

In [ ]:
# probs = np.random.default_rng(0).random(10)
probs = np.ones(20)
print(probs)
np.random.default_rng(0).dirichlet(3e-2*np.ones_like(probs))

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


array([2.94206294e-06, 1.12129286e-43, 1.21628623e-03, 1.11971024e-07,
       1.74142825e-09, 9.98081998e-01, 6.79592227e-04, 6.51977421e-08,
       8.63025820e-07, 1.15352397e-21, 1.03655377e-53, 1.82796448e-10,
       1.62665287e-10, 6.77227710e-10, 3.86905591e-08, 2.88778959e-33,
       3.86453097e-09, 4.51735017e-15, 1.80958917e-05, 2.31040470e-19])

**Values of Opposing Player States**

AlphaZero simulates both sides of a game. Each level of the search tree represents an alternating player's perspective. MCTS's job is to pick the best move for whichever player's turn it is, at a given state in the search tree.

Each `MCSTNode` stores the state value from the perspective of the previous turn's player, and action probabilities from the perspective of the player at that state. This means each node's state and action values are opposite signs. This is because a movement decision considers an action and the state that action leads to.

The job of the neural network is to accurately assess the value of a potential state from the perspective of the player going to that state, and assign action probabilities that reflect the best possible actions to take for the next player at that state. In other words:

> How good is this state for me?

> If I were playing whosever turn it is at this state, how would I rank my options?



:::{.callout-note}
AlphaZero is *not* a neural network that uses MCTS to train. The model and MCTS are integrally linked. Though AlphaZero's model is trained to assess a state and rank available moves, it is meant to work with MCTS.
:::

**Masking**

Simulation selection takes an argmax of the Upper Confidence Bound (UCB) array. AlphaZero needs a way to ignore invalid moves, but if the UCB array is limited to valid moves, its argmax isn't guaranteed to match the index of the corresponding move in the state engine.

There are 2 immediate solutions: 1) either a mapping (`dict`) between the indices of all moves and valid moves can be calculated or stored at each node, or 2) a masking function can be used to perform the mapping on selection.

Masking works by only leaving up for consideration indices that don't result in a true value in a mask array.

In [ ]:
action_array = np.array([1,1,0,2,1,0,0,1])
select_array = np.array([0.1,0.2,0.9,1.2,0.3,0.5,1.8,0.0])
mask = np.zeros(action_array.size)
mask[np.where(action_array==0)] = 1
a = np.argmax(np.ma.array(select_array, mask=mask))
assert action_array[a] == 2